# Code-Defined Geometry Refined MF6 Workflow

This notebook shows a code-first workflow that still feels practical for real model setup.
The important shapes are defined once in code with `shapely`, then reused for:

- `TriangleGrid` refinement
- named model regions for later querying
- package setup for LAK, SFR, DRN, GHB, recharge, and UZF

A useful pattern is to let only the geometries that truly matter for mesh density drive refinement.
Here, the stream corridor and lake shape refine the grid, while the drain/GHB/recharge geometries are still defined in code and reused later for package setup.

Some current package builders still expect GIS files, so this notebook writes temporary GeoPackages from the same in-memory geometries right before package creation. The geometry definitions still live in one place.


In [ ]:
from pathlib import Path
import shutil

import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, Point, Polygon

import myflopy as mf
from myflopy.modflow.mf6.drn import DRNFromVector
from myflopy.modflow.mf6.ghb import GHBFromVector
from myflopy.modflow.mf6.lakes import LAKBuilder
from myflopy.modflow.mf6.recharge import RCHFromVector
from myflopy import SFRBuilder
from myflopy import ModelContext, UZFBuilder
from myflopy.modflow.mf6.simulation.discretization import DisvGrid, TemporalDiscretization
from myflopy.modflow.mf6.simulation.packages import (
    CHD,
    InitialConditions,
    KFlow,
    OutputControl,
    Recharge,
    Storage,
)


In [ ]:
if Path.cwd().name == "notebooks" and Path.cwd().parent.name == "mf6":
    examples_root = Path.cwd().parent
else:
    examples_root = Path.cwd() / "examples" / "mf6"

workspace = examples_root / "artifacts" / "code_geometry_refined_model_workflow"
if workspace.exists():
    shutil.rmtree(workspace, ignore_errors=True)
workspace.mkdir(parents=True, exist_ok=True)

workspace

In [ ]:
def write_features(path: Path, rows: list[dict], crs: str = "EPSG:2927") -> Path:
    gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=crs)
    gdf.to_file(path, driver="GPKG")
    return path


domain_geom = Polygon([(0, 0), (600, 0), (600, 450), (0, 450)])
lake_geom = Point(220, 170).buffer(60)
stream_line = LineString([(50, 360), (260, 270), (540, 140)])
drain_geom = Polygon([(0, 330), (180, 330), (180, 450), (0, 450)])
ghb_geom = Polygon([(0, 0), (180, 0), (180, 120), (0, 120)])
uplands_geom = Polygon([(0, 220), (600, 220), (600, 450), (0, 450)])
lowlands_geom = Polygon([(0, 0), (600, 0), (600, 220), (0, 220)])

domain_geom.area, lake_geom.area, stream_line.length

In [ ]:
triangle_ws = workspace / "triangle_build"
triangle_ws.mkdir(parents=True, exist_ok=True)

tri = mf.TriangleGrid(model_ws=str(triangle_ws))
tri.set_domain_polygon(domain_geom)
tri.add_region_polygon(
    Polygon([(40, 40), (560, 40), (560, 410), (40, 410)]),
    max_area=18000,
    label="mid_refine",
)
tri.add_region_polygon(
    stream_line.buffer(30),
    max_area=4000,
    label="stream_refine",
    priority=2,
    source="line",
)
tri.add_region_polygon(
    lake_geom,
    max_area=1200,
    label="lake_refine",
    priority=3,
    source="circle",
)
tri.build(verbose=False)

vor = mf.VoronoiGridPlus(tri)
top = 115.0 - (0.02 * np.asarray(vor.centroids_x)) + (0.01 * np.asarray(vor.centroids_y))
bottom = top - 30.0
vor.gdf_topbtm = gpd.GeoDataFrame(
    {
        "geometry": vor.gdf_vorPolys.geometry,
        0: top,
        1: bottom,
    },
    geometry="geometry",
    crs=vor.crs,
)

len(vor.gdf_vorPolys)

In [ ]:
tri.preview_regions()[["label", "max_area", "claim_area", "priority"]]

In [ ]:
vor.plot2d()

In [ ]:
model = mf.SimulationBase(name="geom_refined", mf_folder_path=workspace, vor=vor, nper=1)
DisvGrid(vor=vor, model=model, top=top.tolist(), bottom=[bottom.tolist()], nlay=1)
TemporalDiscretization(model=model, per_len=1, num_steps=1, multiplier=1.0)
InitialConditions(model=model, vor=vor, nlay=1, strt=(top - 5.0).tolist())
KFlow(model=model, k=[15.0] * vor.ncpl, save_specific_discharge=False)
Storage(model=model, sto_steady={0: True}, sto_transient={})
OutputControl(model=model)

east_strip = Polygon([(560, 0), (600, 0), (600, 450), (560, 450)])
east_cells = sorted(set(vor.get_vor_cells_as_series(east_strip).iloc[0]))
CHD(model=model, stress_period_data={0: [[(0, cell), 100.0] for cell in east_cells]})

model.add_region_from_geometry("lake_circle_region", lake_geom, category="custom", overwrite=True)
model.add_region_from_geometry("stream_corridor_region", stream_line.buffer(30), category="custom", overwrite=True)
model.list_regions().tail()

In [ ]:
drain_path = write_features(
    workspace / "drain.gpkg",
    [{"name": "northwest_drain", "height": 2.0, "cond": 500.0, "layer": 1, "min_elev": 80.0, "geometry": drain_geom}],
    crs=vor.crs,
)
ghb_path = write_features(
    workspace / "ghb.gpkg",
    [{"name": "southwest_ghb", "elev": 90.0, "height": 0.0, "cond": 650.0, "layer": 1, "min_elev": 78.0, "geometry": ghb_geom}],
    crs=vor.crs,
)
recharge_path = write_features(
    workspace / "recharge.gpkg",
    [
        {"zone": "uplands", "rch_0": 0.0020, "geometry": uplands_geom},
        {"zone": "lowlands", "rch_0": 0.0015, "geometry": lowlands_geom},
    ],
    crs=vor.crs,
)
lake_path = write_features(
    workspace / "lake.gpkg",
    [{"name": "lake_0", "geometry": lake_geom}],
    crs=vor.crs,
)
stream_path = write_features(
    workspace / "stream.gpkg",
    [{"name": "stream_0", "geometry": stream_line}],
    crs=vor.crs,
)

drain_path, lake_path, stream_path

In [ ]:
drn_builder = DRNFromVector(model=model, vor=vor, shp_gpkg=drain_path, uid="name", idomain=[1] * vor.ncpl)
drn_dict = drn_builder.from_vector(
    edges_only=True,
    register_regions=True,
    region_name_prefix="drn_group",
    combined_region_name="all_drains",
    region_tags=["drn"],
    overwrite_regions=True,
)
mf.modflow.mf6.simulation.packages.Drains(model=model, stress_period_data=drn_dict)

ghb_builder = GHBFromVector(model=model, vor=vor, shp_gpkg=ghb_path, uid="name", idomain=[1] * vor.ncpl)
ghb_dict = ghb_builder.from_vector(
    register_regions=True,
    region_name_prefix="ghb_group",
    combined_region_name="all_ghb",
    region_tags=["ghb"],
    overwrite_regions=True,
)
mf.modflow.mf6.simulation.packages.GHB(model=model, stress_period_data=ghb_dict)

recharge_builder = RCHFromVector(
    model=model,
    vor=vor,
    shp_gpkg=recharge_path,
    uid="zone",
    rch_fields=["rch_0"],
    rch_fields_to_pers=[0],
    background_rch=0.0,
    grid_type="disv",
    limit_to_k33=False,
)
rch_dict = recharge_builder.from_vector(
    register_regions=True,
    region_name_prefix="rch_zone",
    combined_region_name="all_rch",
    region_tags=["rch"],
    overwrite_regions=True,
)
Recharge(model=model, vor=vor, rch_dict=rch_dict)

uzf_context = ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain))
uzf_cells = [(0, cell) for cell in range(vor.ncpl)]
uzf_finf = {period: [{tuple(row[0]): row[1] for row in rows}.get(cellid, 0.0) for cellid in uzf_cells] for period, rows in rch_dict.items()}
uzf = UZFBuilder(
    context=uzf_context,
    nper=model.nper,
    cells=uzf_cells,
    vks=0.5,
    thtr=0.1,
    thts=0.3,
    thti=0.2,
    finf=uzf_finf,
)
uzf.build().build(model.gwf)
model.add_region_from_cells("uzf_all", uzf.uzf_cells, category="boundary", package="uzf", tags=["uzf"], overwrite=True)

lak = LAKBuilder(
    context=ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain)),
    nper=model.nper,
    lakes=[lake_path],
    lake_id_field="name",
    starting_stage=100.0,
    lake_bottom=92.0,
    bed_leakance=0.05,
    connection_modes="automatic",
    status="ACTIVE",
    mover=False,
)
lak.build().build(model.gwf)
for lake_id, cells in lak.lake_cells.items():
    model.add_region_from_cells(f"lake_zone_{lake_id}", [(0, cell) for cell in cells], category="boundary", package="lak", tags=["lak"], geometry=lak.lake_table.loc[lake_id].geometry, overwrite=True)
model.add_region_from_cells("all_lakes", [(0, cell) for cells in lak.lake_cells.values() for cell in cells], category="boundary", package="lak", tags=["lak"], overwrite=True)

sfr = SFRBuilder(
    context=ModelContext(grid=vor, domain=np.asarray(model.gwf.modelgrid.idomain)),
    nper=model.nper,
    streams=[stream_path],
    width=12.0,
    gradient=0.001,
    roughness=0.03,
    streambed_k=2.0,
    streambed_thickness=1.5,
)
sfr.build().build(model.gwf)
model.add_region_from_cells("all_streams", [(0, cell) for cells in sfr.stream_cells.values() for cell in cells], category="boundary", package="sfr", tags=["sfr"], overwrite=True)

model.add_group(
    "boundary_features",
    members=["all_drains", "all_ghb", "all_rch", "uzf_all", "all_lakes", "all_streams"],
    overwrite=True,
)


In [ ]:
success, _ = model.run_simulation()
success

In [ ]:
model.list_regions().tail(10)

In [ ]:
resolved_cells, trace = model.resolve_region_cells_with_trace("boundary_features")
len(resolved_cells), list(trace.items())[:5]

In [ ]:
model.region_heads("boundary_features", per=0)[["cell", "elev", "region"]].head()

In [ ]:
model.cor(per=0)